# Тестирование OCR Service

Этот ноутбук симулирует работу сервиса:
1. Создаёт запись в БД со статусом `pending`
2. Загружает изображение в БД
3. Отправляет задачу в RabbitMQ очередь `text_recognition`

In [2]:
import asyncpg
import aio_pika
import asyncio
import uuid
from pathlib import Path

# Параметры подключения
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "certificates"
DB_USER = "postgres"
DB_PASS = "postgres"

RABBIT_HOST = "localhost"
RABBIT_PORT = 5672
RABBIT_USER = "guest"
RABBIT_PASS = "guest"
QUEUE_NAME = "text_recognition"

## Шаг 1: Загрузка изображения в БД

In [25]:
async def upload_image_to_db(image_path: str, mime_type: str = "image/jpeg"):
    """Загружает изображение в БД и возвращает ID записи."""
    
    # Читаем файл
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    print(f"📂 Loaded image: {image_path}, size={len(image_data)} bytes")
    
    # Подключаемся к БД
    conn = await asyncpg.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
    )
    
    # Генерируем UUID
    task_id = str(uuid.uuid4())
    filename = Path(image_path).name
    
    # Вставляем запись
    await conn.execute(
        """
        INSERT INTO certificates (
            id, image_data, image_filename, image_mime_type, 
            image_size_bytes, status, created_at, updated_at
        ) VALUES ($1, $2, $3, $4, $5, $6, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
        """,
        task_id,
        image_data,
        filename,
        mime_type,
        len(image_data),
        "pending",
    )
    
    print(f"✅ Created record in DB: id={task_id}, status=pending")
    
    await conn.close()
    return task_id

# Загружаем test_1.jpg
TASK_ID = await upload_image_to_db("test_1.jpg", "image/jpeg")
print(f"\n📝 Task ID: {TASK_ID}")

📂 Loaded image: test_1.jpg, size=834843 bytes
✅ Created record in DB: id=7df81a1a-5018-4d31-bea8-811e61ea5e1c, status=pending

📝 Task ID: 7df81a1a-5018-4d31-bea8-811e61ea5e1c


## Шаг 2: Отправка задачи в RabbitMQ

In [26]:
async def send_task_to_queue(task_id: str):
    """Отправляет задачу в очередь RabbitMQ."""
    
    # Подключаемся к RabbitMQ
    connection = await aio_pika.connect(
        host=RABBIT_HOST,
        port=RABBIT_PORT,
        login=RABBIT_USER,
        password=RABBIT_PASS,
    )
    
    async with connection:
        channel = await connection.channel()
        
        # Объявляем очередь
        queue = await channel.declare_queue(QUEUE_NAME, durable=True)
        
        # Формируем сообщение
        message_body = f'{{"id": "{task_id}"}}'.encode()
        
        # Отправляем через default exchange
        await channel.default_exchange.publish(
            aio_pika.Message(
                body=message_body,
                delivery_mode=aio_pika.DeliveryMode.PERSISTENT,
            ),
            routing_key=QUEUE_NAME,
        )
        
        print(f"✅ Sent task to queue: {task_id}")
    
    return task_id

# Отправляем задачу
await send_task_to_queue(TASK_ID)

✅ Sent task to queue: 7df81a1a-5018-4d31-bea8-811e61ea5e1c


'7df81a1a-5018-4d31-bea8-811e61ea5e1c'

## Шаг 3: Проверка статуса (после обработки)

Запусти `main.py` и выполни этот блок для проверки результата:

In [5]:
async def check_task_status(task_id: str):
    """Проверяет статус задачи в БД."""
    
    conn = await asyncpg.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
    )
    
    row = await conn.fetchrow(
        """
        SELECT id, status, raw_text, error_message, completed_at
        FROM certificates
        WHERE id = $1
        """,
        task_id,
    )
    
    if row:
        print(f"📋 Task Status:")
        print(f"   ID: {row['id']}")
        print(f"   Status: {row['status']}")
        print(f"   Completed: {row['completed_at']}")
        
        if row['raw_text']:
            print(f"\n📝 Recognized Text ({len(row['raw_text'])} chars):")
            print(f"   {row['raw_text'][:200]}..." if len(row['raw_text']) > 200 else f"   {row['raw_text']}")
        
        if row['error_message']:
            print(f"\n❌ Error: {row['error_message']}")
    else:
        print(f"❌ Task not found: {task_id}")
    
    await conn.close()

# Проверка (выполнить после обработки)
await check_task_status(TASK_ID)

📋 Task Status:
   ID: 49e44ca0-b610-444f-a4e8-d20dfd41d3d5
   Status: pending
   Completed: None
